In [1]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importamos nuestro módulo de ML
ml = importlib.import_module("flows.05_ml_models")
sns.set_theme(style="whitegrid", context="notebook")

# Cargamos el dataset formulado
X, y = ml.load_dataset()
print("Dimensiones de X:", X.shape)
print("Tasa de supervivencia (y.mean()):", round(y.mean(), 4))

Dimensiones de X: (891, 12)
Tasa de supervivencia (y.mean()): 0.3838


# Formulación del Problema de Machine Learning (Paso 39)

| Decisión | Tu respuesta |
| :--- | :--- |
| **Tipo de problema** | Clasificación binaria supervisada |
| **Variable objetivo** | `Survived` (1 = sobrevivió) |
| **Unidad de análisis y momento de predicción** | Un pasajero, *al embarcar* |
| **Métrica principal** | ROC-AUC, porque evalúa la capacidad de discriminación sin depender de un umbral de decisión rígido en un entorno desbalanceado (~38% positivos). |
| **Métricas secundarias** | precisión, recall, exactitud, F1-score |
| **Costo de errores** | Para una aseguradora, predecir falsamente que alguien sobrevive genera un riesgo financiero crítico de estimación; predecir que muere representa una oportunidad comercial perdida. |
| **Criterio de éxito** | Superar el baseline por un margen que puedas defender |


## Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe antes del evento? | ¿Riesgo de fuga? | Decisión |
| :--- | :--- | :--- | :--- |
| `Survived`, `Survivor` | No (es el objetivo) | **Fuga directa** | **Excluir** |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores: memorizan, no generalizan | **Excluir** (el título sí, vía `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican a `Sex`, etc. con un orden arbitrario | **Excluir** (usar *one-hot*) |
| `Age` imputada | — | Estadística calculada con todos los datos | **Usar `AgeOriginal`** |
| `Cabin`, `TCabin`, `HasCabin` | **Dudoso** | ¿Se registró la cabina *después* del naufragio (más fácil para los sobrevivientes)? | **Evaluar con y sin ella** (Paso 61) |
| `Person` | Derivada de Age / Sex | Redundante con ambas | **Excluir del modelo base** |

**Reflexión sobre el debate de `HasCabin`:**
Si `HasCabin` mejora la métrica de predicción, no significa necesariamente que tener cabina otorgue una ventaja física de supervivencia. Existe un **artefacto de registro**: la información de la cabina se conservó y documentó con mayor precisión para los pasajeros que sobrevivieron y pudieron dar testimonio, inflando artificialmente el valor predictivo de esta variable. Un modelo robusto debe evaluar este tipo de variables con cautela para no aprender sesgos burocráticos en lugar de riesgo real.